In [5]:
import numpy as np
import json
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModel


class OmniNAEmbeddingExtractor:
    """
    Извлекает эмбеддинги для последовательностей ДНК/РНК из модели OmniNA.
    Использует среднее по токенам скрытых состояний последнего слоя.
    """
    def __init__(self, model_name: str = "XLS/OmniNA-66m", device: str = None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        # Инициализация токенизатора и установка токена паддинга
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        if self.tokenizer.pad_token is None:
            # для causal LM используем eos_token в качестве pad_token
            self.tokenizer.pad_token = self.tokenizer.eos_token
        # Загружаем модель с возвратом скрытых состояний
        self.model = AutoModel.from_pretrained(
            model_name,
            output_hidden_states=True
        ).to(self.device)
        # Обеспечиваем согласованность pad_token_id в конфиге
        if self.model.config.pad_token_id is None:
            self.model.config.pad_token_id = self.model.config.eos_token_id
        self.model.eval()

    def extract_embeddings(self, sequences, batch_size: int = 8):
        """
        Возвращает numpy-массив эмбеддингов формы (len(sequences), hidden_size).
        """
        all_embs = []
        with torch.no_grad():
            for i in range(0, len(sequences), batch_size):
                batch = sequences[i: i + batch_size]
                enc = self.tokenizer(
                    batch,
                    return_tensors="pt",
                    padding=True,
                    truncation=True
                ).to(self.device)
                out = self.model(**enc)
                # Последнее скрытое состояние: (batch, seq_len, hidden)
                hidden = out.hidden_states[-1]
                # mean pooling по токенам (учёт маски паддинга)
                mask = enc.attention_mask.unsqueeze(-1)
                summed = (hidden * mask).sum(dim=1)
                lengths = mask.sum(dim=1)
                embs = (summed / lengths).cpu().numpy()
                all_embs.append(embs)
        return np.vstack(all_embs)


ds = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks")
train_ds, test_ds = ds['train'], ds['test']

extractor = OmniNAEmbeddingExtractor()
PARAMS_LOGREG = {'max_iter': 1000, 'random_state': 42}
PATH_TO_SAVE_OUTPUTS = '.'
BATCH_SIZE = 16

# Baseline: обучение на полном наборе
baseline = {}
for task in tqdm(set(train_ds['task']), desc='Baseline'):
    tr = train_ds.filter(lambda x, t=task: x['task'] == t)
    te = test_ds.filter(lambda x, t=task: x['task'] == t)
    seqs_tr, y_tr = tr['sequence'], np.array(tr['label'])
    seqs_te, y_te = te['sequence'], np.array(te['label'])

    X_tr = extractor.extract_embeddings(seqs_tr, batch_size=BATCH_SIZE)
    X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)

    clf = LogisticRegression(**PARAMS_LOGREG)
    Xf = X_tr.reshape(-1, 1) if (X_tr.ndim == 1 or X_tr.shape[1] == 1) else X_tr
    Xt = X_te.reshape(-1, 1) if (X_te.ndim == 1 or X_te.shape[1] == 1) else X_te
    clf.fit(Xf, y_tr)
    preds = clf.predict(Xt)

    baseline[task] = {
        'accuracy': float(accuracy_score(y_te, preds)),
        'f1_score': float(f1_score(y_te, preds, average='macro'))
    }
    with open(f'{PATH_TO_SAVE_OUTPUTS}/results_omnina_task-{task}_baseline.json', 'w') as f:
        json.dump(baseline, f, indent=4)

# Few-shot эксперименты
def few_shot(train, test, ks=(1, 5, 10, 20), trials=5):
    res = {}
    rng = np.random.RandomState(42)
    for task in tqdm(set(train['task']), desc='Few-shot'):
        tr = train.filter(lambda x, t=task: x['task'] == t)
        te = test.filter(lambda x, t=task: x['task'] == t)
        seqs_tr, y_tr = tr['sequence'], np.array(tr['label'])
        seqs_te, y_te = te['sequence'], np.array(te['label'])
        X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)
        res[task] = {}
        for k in ks:
            accs, f1s = [], []
            for _ in range(trials):
                idxs = []
                for lbl in np.unique(y_tr):
                    locs = np.where(y_tr == lbl)[0]
                    choice = rng.choice(locs, size=min(k, len(locs)), replace=False)
                    idxs.extend(choice.tolist())
                X_k = extractor.extract_embeddings([seqs_tr[i] for i in idxs], batch_size=BATCH_SIZE)
                y_k = y_tr[idxs]
                clf = LogisticRegression(**PARAMS_LOGREG)
                Xf = X_k.reshape(-1, 1) if (X_k.ndim == 1 or X_k.shape[1] == 1) else X_k
                Xt = X_te.reshape(-1, 1) if (X_te.ndim == 1 or X_te.shape[1] == 1) else X_te
                clf.fit(Xf, y_k)
                p = clf.predict(Xt)
                accs.append(accuracy_score(y_te, p))
                f1s.append(f1_score(y_te, p, average='macro'))
            res[task][k] = {'accuracy': float(np.mean(accs)), 'f1_score': float(np.mean(f1s))}
            with open(f'results_omnina_task-{task}_k-{k}.json', 'w') as f:
                json.dump(res, f, indent=4)
    return res


results_kshot = few_shot(train_ds, test_ds)

output = {'full': baseline, 'kshot': results_kshot, 'params': PARAMS_LOGREG}
with open(f'{PATH_TO_SAVE_OUTPUTS}/results_omnina.json', 'w') as f:
    json.dump(output, f, indent=4)


/home/mikhail_nuridinov/notebooks/models/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.
2025-07-19 19:17:48.030903: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for